# Extracción de características

**Objetivo:** transformar las actividades limpias en **características mensuales** por estudiante, de modo que un modelo entrenado con la información disponible hasta un mes determinado pueda predecir el resultado del curso (detección temprana).

| | |
|---|---|
| **Entradas** | `Datos/actividades_cleaned.csv` y `Datos/registro_cleaned.csv` |
| **Salida** | `Datos/notas_pensamiento_cuantitativo.csv` (2,325 filas × 41 columnas, una fila por estudiante-intento) |
| **Siguiente paso** | [`../exploratorio/analisis_exploratorio.ipynb`](../exploratorio/analisis_exploratorio.ipynb) |

**Idea principal:** el semestre va de enero a mayo. Para cada mes *k* calculamos qué tan bien le ha ido al estudiante **hasta ese mes**. Las características mensuales solo usan información disponible hasta el mes *k*, es decir, lo que un catedrático conocería en ese momento.

### Características generadas

| Variable | Descripción |
|---|---|
| `porcentaje_acum_actividades_mes_k` | % de puntos obtenidos sobre los posibles en actividades (sin exámenes), acumulado hasta el mes *k* |
| `entregas_tarde_acum_mes_k` | Número de entregas tardías acumuladas hasta el mes *k* |
| `nota_examen_mes_k` | Nota (en %) del examen del mes *k*: Parcial 1 → febrero, Parcial 2 → marzo, Parcial 3 → abril, Parcial 4 / Examen final → mayo |
| `delta_actividades_mes_k` | Cambio del porcentaje acumulado en actividades respecto al mes anterior |
| `z_porcentaje_acum_actividades_mes_k`, `z_nota_examen_mes_k` | Puntuación z dentro de la misma sección y año (desempeño relativo a los compañeros) |
| `CantComentarios` | Total de comentarios recibidos en las entregas |
| `DiferenciaPromedio` | Mediana de días entre la entrega y la calificación (velocidad de calificación) |
| `Virtualidad` | 1 si el curso se impartió en modalidad virtual (2020 y 2021), 0 si fue presencial |

Además, `Colegio` y `TituloNivelMedio` se simplifican agrupando las categorías poco frecuentes en `OTRO`.

In [1]:
# Librerías
import pandas as pd
import math
import numpy as np
from sklearn.preprocessing import TargetEncoder

In [2]:
# Carga de actividades y registro limpios
actividades = pd.read_csv('../../Datos/actividades_cleaned.csv')
df_registro = pd.read_csv('../../Datos/registro_cleaned.csv')

Revisamos las dimensiones y las columnas de ambos conjuntos.

In [3]:
# Dimensiones del registro
df_registro.shape

(2325, 10)

In [4]:
# Columnas de actividades
actividades.columns

Index(['Est_ID', 'Actividad', 'Seccion', 'Ciclo', 'zona', 'Anio',
       'TipoCalificacion', 'PuntosPosibles', 'FechaTodoElDia',
       'FechaVencimiento', 'TipoEntrega', 'Nota', 'FechaCalificacion',
       'FechaEnvio', 'TipoEnvio', 'CantComentarios', 'NotaPtos',
       'NotaPorciento', 'imputado', 'residuo_estudiante', 'entrega_tarde'],
      dtype='object')

In [5]:
# Columnas del registro
df_registro.columns

Index(['Est_ID', 'PAA', 'Colegio', 'TituloNivelMedio', 'NotaFinal', 'Aprobado',
       'Genero', 'EstadoCurso', 'veces_curso', 'Anio'],
      dtype='object')

### Identificación de exámenes
Marcamos como examen los parciales 1 a 4 y el examen final. Los separamos del resto de actividades porque miden algo distinto (evaluaciones sumativas en una fecha fija) y porque queremos una característica propia para la nota de cada examen.

In [6]:
def es_examen(texto):
    """Devuelve 1 si la actividad es un parcial o el examen final"""
    if texto in ['PARCIAL 1', 'PARCIAL 2', 'PARCIAL 3', 'PARCIAL 4', 'EXAMEN FINAL']:
        return 1
    return 0

In [7]:
# Marca de exámenes
actividades['es_examen'] = actividades['Actividad'].apply(es_examen)

Cada examen se asigna al mes en que se realiza según el calendario del curso: Parcial 1 → febrero (2), Parcial 2 → marzo (3), Parcial 3 → abril (4), Parcial 4 y examen final → mayo (5).

In [8]:
def asignar_mes_a_examen(nombre_examen):
    """Devuelve el mes del semestre en que se realiza cada examen"""
    if nombre_examen == 'PARCIAL 1':
        return 2
    elif nombre_examen == 'PARCIAL 2':
        return 3
    elif nombre_examen == 'PARCIAL 3':
        return 4
    elif nombre_examen == 'PARCIAL 4' or nombre_examen == 'EXAMEN FINAL':
        return 5
    
    

### Asignación del mes de cada actividad
- El mes se toma de `FechaVencimiento`; si no existe, de `FechaTodoElDia` y, en último caso, de `FechaCalificacion`.
- Las actividades de **junio** se agrupan con **mayo** (mes 5): en junio solo quedan los últimos días del semestre.
- Las actividades sin ninguna fecha se asignan al mes 5. Es una decisión conservadora: así no se usan en los modelos de los primeros meses información que podría ser del final del semestre.

In [9]:
# Mes de cada actividad a partir de sus fechas
for col in ['FechaVencimiento', 'FechaTodoElDia', 'FechaCalificacion', 'FechaEnvio']:
    actividades[col] = pd.to_datetime(
        actividades[col],
        format='mixed',
        errors='coerce'
    )

actividades['fecha_para_mes'] = (
    actividades['FechaVencimiento']
    .fillna(actividades['FechaTodoElDia'])
    .fillna(actividades['FechaCalificacion'])
)

actividades['mes'] = actividades['fecha_para_mes'].dt.month
# Junio se agrupa con mayo
actividades['mes'] = actividades['mes'].replace(6, 5)  


# Actividades sin fecha se asignan a mayo
actividades['mes'] = actividades['mes'].fillna(5)

Para los exámenes, el mes del calendario (celda anterior) reemplaza al mes calculado con las fechas, ya que muchos exámenes no tienen fecha registrada.

In [10]:
# Los exámenes toman el mes del calendario del curso
es_ex = actividades['es_examen'] == 1

mes_ex = actividades.loc[es_ex, 'Actividad'].apply(asignar_mes_a_examen)
actividades.loc[es_ex, 'mes'] = mes_ex.fillna(actividades.loc[es_ex, 'mes'])
actividades['mes'] = actividades['mes'].astype(int)

In [11]:
# Columnas actuales
actividades.columns

Index(['Est_ID', 'Actividad', 'Seccion', 'Ciclo', 'zona', 'Anio',
       'TipoCalificacion', 'PuntosPosibles', 'FechaTodoElDia',
       'FechaVencimiento', 'TipoEntrega', 'Nota', 'FechaCalificacion',
       'FechaEnvio', 'TipoEnvio', 'CantComentarios', 'NotaPtos',
       'NotaPorciento', 'imputado', 'residuo_estudiante', 'entrega_tarde',
       'es_examen', 'fecha_para_mes', 'mes'],
      dtype='object')

Asignamos a cada estudiante-año su sección (la primera registrada). La sección se usa para calcular las puntuaciones z.

In [12]:
# Sección de cada estudiante por año
seccion_estudiante = (
    actividades
    .groupby(['Est_ID', 'Anio'])['Seccion']
    .first() 
    .reset_index()
)

df_registro = df_registro.merge(seccion_estudiante, on=['Est_ID', 'Anio'], how='left')

### Características mensuales
En esta celda:
1. Separamos los puntos de actividades (`ptos_act`) de los de exámenes, para que el porcentaje de actividades no se mezcle con las notas de exámenes.
2. Agregamos por estudiante, año, sección y mes, y calculamos **acumulados**: el porcentaje acumulado de actividades y las entregas tardías acumuladas. Usamos acumulados (y no el valor aislado de cada mes) porque resumen todo lo que se sabe del estudiante hasta ese momento y son más estables.
3. Pasamos de formato largo a ancho (una columna por mes). Si un estudiante no tiene registros en un mes, se arrastra el valor del mes anterior (`ffill`), porque su acumulado no cambió.
4. Calculamos la nota de cada examen en porcentaje (sin `ffill`: si no hay examen ese mes, queda vacío).
5. Calculamos el **delta** mensual del porcentaje de actividades, para capturar si el estudiante mejora o empeora.
6. Calculamos **puntuaciones z** dentro de cada (año, sección), para medir el desempeño de cada estudiante relativo a sus compañeros de sección.
7. Verificamos que ningún estudiante esté en más de una sección el mismo año y unimos todo al registro.

In [ ]:
# Características mensuales acumuladas por estudiante
claves = ['Est_ID', 'Anio', 'Seccion']
es_ex = actividades['es_examen'] == 1

# Encontramos puntos de actividades y de exámenes
actividades['ptos_act'] = actividades['NotaPtos'].where(~es_ex, 0)
actividades['posibles_act'] = actividades['PuntosPosibles'].where(~es_ex, 0)

# Agrupamos por mes
mensual = (
    actividades
    .groupby(claves + ['mes'], as_index=False)
    .agg(
        NotaPtos_mes=('NotaPtos', 'sum'),
        Nota_posible_mes=('PuntosPosibles', 'sum'),
        ptos_act_mes=('ptos_act', 'sum'),
        posibles_act_mes=('posibles_act', 'sum'),
        entregas_tarde_mes=('entrega_tarde', 'sum'),
    )
    .sort_values(claves + ['mes'])
)

g = mensual.groupby(claves)

# Porcentaje acumulado en actividades y entregas tardías acumuladas
mensual['porcentaje_acum_actividades'] = (
    g['ptos_act_mes'].cumsum() / g['posibles_act_mes'].cumsum().replace(0, np.nan) * 100
)
mensual['entregas_tarde_acum'] = g['entregas_tarde_mes'].cumsum()

def a_ancho(df, valor, prefijo, ffill=True):
    """Pasa una variable mensual de formato largo a ancho, una columna por mes"""
    p = df.pivot_table(index=claves, columns='mes', values=valor)
    if ffill:
        p = p.ffill(axis=1)   # mes sin registros
    p.columns = [f'{prefijo}_mes_{int(m)}' for m in p.columns]
    return p

pivot_act = a_ancho(mensual, 'porcentaje_acum_actividades', 'porcentaje_acum_actividades')
pivot_tarde = a_ancho(mensual, 'entregas_tarde_acum', 'entregas_tarde_acum')

# Calculamos la nota en cada examen del mes
ex = (
    actividades[es_ex]
    .groupby(claves + ['mes'], as_index=False)
    .agg(ptos=('NotaPtos', 'sum'), posibles=('PuntosPosibles', 'sum'))
)
ex['nota_examen'] = ex['ptos'] / ex['posibles'].replace(0, np.nan) * 100
pivot_ex = a_ancho(ex, 'nota_examen', 'nota_examen', ffill=False)

# Unión de las características mensuales
features_mensuales = pivot_act.join([pivot_tarde, pivot_ex], how='outer')

# Calculamos el cambio en el porcentaje acumulado de actividades con respecto al mes anterior
meses_act = sorted(int(c.rsplit('_', 1)[1]) for c in pivot_act.columns)
for anterior, actual in zip(meses_act[:-1], meses_act[1:]):
    features_mensuales[f'delta_actividades_mes_{actual}'] = (
        features_mensuales[f'porcentaje_acum_actividades_mes_{actual}']
        - features_mensuales[f'porcentaje_acum_actividades_mes_{anterior}']
    )

# Calculamos un z-score dentro de la sección
def zscore_seccion(serie):
    """Calcula la puntuación z de cada estudiante dentro de su año y sección"""
    grupos = serie.groupby(level=['Anio', 'Seccion'])
    std = grupos.transform('std')
    return (serie - grupos.transform('mean')) / std.where(std > 0)

cols_z = [c for c in features_mensuales.columns
          if c.startswith(('porcentaje_acum_actividades_mes_', 'nota_examen_mes_'))]
for c in cols_z:
    features_mensuales[f'z_{c}'] = zscore_seccion(features_mensuales[c])

features_mensuales = features_mensuales.reset_index()

# Cada estudiante debe estar en una sola sección por año
assert not features_mensuales.duplicated(['Est_ID', 'Anio']).any(), \
    "Hay estudiantes en más de una sección el mismo año"

# Unión con el registro
df_registro = df_registro.merge(
    features_mensuales.drop(columns='Seccion'), on=['Est_ID', 'Anio'], how='left'
)

Sumamos la cantidad de comentarios que recibió cada estudiante en sus entregas, como indicador de la retroalimentación recibida.

In [14]:
# Total de comentarios recibidos por estudiante
comentarios = actividades.groupby(['Est_ID', 'Anio'], as_index=False)['CantComentarios'].sum()
df_registro = df_registro.merge(comentarios, on=['Est_ID', 'Anio'], how='left')

Inspección intermedia del resultado.

In [15]:
# Tipos de dato de actividades
actividades.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 68159 entries, 0 to 68158
Data columns (total 26 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Est_ID              68159 non-null  float64       
 1   Actividad           68159 non-null  object        
 2   Seccion             68159 non-null  int64         
 3   Ciclo               68159 non-null  int64         
 4   zona                68159 non-null  bool          
 5   Anio                68159 non-null  int64         
 6   TipoCalificacion    68159 non-null  object        
 7   PuntosPosibles      68159 non-null  float64       
 8   FechaTodoElDia      30848 non-null  datetime64[ns]
 9   FechaVencimiento    51359 non-null  datetime64[ns]
 10  TipoEntrega         68159 non-null  object        
 11  Nota                68159 non-null  float64       
 12  FechaCalificacion   68159 non-null  datetime64[ns]
 13  FechaEnvio          48960 non-null  datetime64

### Velocidad de calificación (`DiferenciaPromedio`)
Calculamos, para cada estudiante, la **mediana** de días entre la entrega (`FechaEnvio`) y la calificación (`FechaCalificacion`) de sus actividades, sin contar exámenes. A pesar del nombre de la columna se usa la mediana, por ser robusta a valores atípicos.

In [16]:
# Velocidad de calificación: mediana de días entre la entrega y la calificación
mask = ~actividades['Actividad'].str.contains(
    r'parcial|examen|evaluación|evaluacion',
    case=False,
    na=False
)

velocidad_calificacion = (
    actividades.loc[mask]
    .assign(
        DifDias=lambda df: (
            df['FechaCalificacion'] - df['FechaEnvio']
        ).dt.days
    )
    .groupby(['Est_ID', 'Anio', 'Seccion'], as_index=False)
    .agg(DiferenciaPromedio=('DifDias', 'median'))
)

df_registro = df_registro.merge(
    velocidad_calificacion,
    on=['Est_ID', 'Anio'],
    how='left'
)

Inspecciones intermedias del dataset resultante.

In [17]:
# Estudiantes con velocidad de calificación positiva
df_registro[df_registro['DiferenciaPromedio'] > 0]

,Est_ID,PAA,Colegio,TituloNivelMedio,NotaFinal,Aprobado,Genero,EstadoCurso,veces_curso,Anio,...,z_porcentaje_acum_actividades_mes_3,z_porcentaje_acum_actividades_mes_4,z_porcentaje_acum_actividades_mes_5,z_nota_examen_mes_2,z_nota_examen_mes_3,z_nota_examen_mes_4,z_nota_examen_mes_5,CantComentarios,Seccion_y,DiferenciaPromedio
482,2457.0,1414.0,COLEGIO VON HUMBOLDT,BACHILLER EN CIENCIAS Y LETRAS,74.0,1,MASCULINO,APROBADO,1,2020,...,-0.775530,-0.978858,-0.949318,-0.820870,-0.420188,-0.468325,0.023767,1.0,10,5.0
483,1050.0,1407.0,COLEGIO METROPOLITANO,BACHILLER EN CIENCIAS Y LETRAS,89.0,1,MASCULINO,APROBADO,1,2020,...,0.618699,0.698314,0.497763,0.670767,0.833931,-0.066876,-0.074209,0.0,100,7.0
484,2296.0,1283.0,COLEGIO INTERNACIONAL MONTESSORI,BACHILLER EN CIENCIAS Y LETRAS,87.0,1,MASCULINO,APROBADO,1,2020,...,0.930196,0.318620,0.414024,-0.936726,1.114659,0.398005,-1.156073,2.0,100,6.0
485,1593.0,1304.0,COLEGIO JULIO VERNE,BACHILLER EN CIENCIAS Y LETRAS,88.0,1,MASCULINO,APROBADO,1,2020,...,0.088471,0.494890,0.549340,-0.184926,-0.146197,0.917029,-0.062199,1.0,100,7.0
486,2266.0,1306.0,COLEGIO MONTANO PORTAL LOS ALAMOS,BACHILLER EN CIENCIAS Y LETRAS CON ORIENTACION...,85.0,1,MASCULINO,APROBADO,1,2020,...,-1.164454,-0.468123,-0.375697,0.556626,-0.311911,-0.191865,-0.200086,0.0,100,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2320,2668.0,1376.0,INSTITUTO AUSTRIACO GUATEMALTECO,BACHILLER EN CIENCIAS Y LETRAS,71.0,1,MASCULINO,APROBADO,2,2022,...,-1.109648,-1.634516,-1.999672,0.566379,0.950765,0.314019,0.512698,14.0,140,7.5
2321,2673.0,1060.0,LICEO MIXTO BILINGUE MILENEO AMERICANO,BACHILLER EN CIENCIAS Y LETRAS,27.0,0,MASCULINO,REPROBADO,1,2020,...,-3.675043,-3.508557,-3.285454,-2.675784,-2.658571,-1.996755,-2.649664,16.0,160,7.0
2322,2673.0,1060.0,LICEO MIXTO BILINGUE MILENEO AMERICANO,BACHILLER EN CIENCIAS Y LETRAS,69.0,1,MASCULINO,APROBADO,2,2021,...,-0.174285,-0.147021,-0.048990,-0.641863,-0.106163,-0.168361,0.059439,4.0,190,4.5
2323,2713.0,1234.0,COLEGIO SAN FRANCISCO JAVIER DE QUICHE,BACHILLER EN CIENCIAS Y LETRAS,43.0,0,MASCULINO,REPROBADO,1,2021,...,-2.334781,-2.564384,-2.673100,-2.480935,-1.480971,-1.933525,-2.038149,4.0,30,4.0


In [18]:
# Columnas del registro
df_registro.columns

Index(['Est_ID', 'PAA', 'Colegio', 'TituloNivelMedio', 'NotaFinal', 'Aprobado',
       'Genero', 'EstadoCurso', 'veces_curso', 'Anio', 'Seccion_x',
       'porcentaje_acum_actividades_mes_1',
       'porcentaje_acum_actividades_mes_2',
       'porcentaje_acum_actividades_mes_3',
       'porcentaje_acum_actividades_mes_4',
       'porcentaje_acum_actividades_mes_5', 'entregas_tarde_acum_mes_1',
       'entregas_tarde_acum_mes_2', 'entregas_tarde_acum_mes_3',
       'entregas_tarde_acum_mes_4', 'entregas_tarde_acum_mes_5',
       'nota_examen_mes_2', 'nota_examen_mes_3', 'nota_examen_mes_4',
       'nota_examen_mes_5', 'delta_actividades_mes_2',
       'delta_actividades_mes_3', 'delta_actividades_mes_4',
       'delta_actividades_mes_5', 'z_porcentaje_acum_actividades_mes_1',
       'z_porcentaje_acum_actividades_mes_2',
       'z_porcentaje_acum_actividades_mes_3',
       'z_porcentaje_acum_actividades_mes_4',
       'z_porcentaje_acum_actividades_mes_5', 'z_nota_examen_mes_2',
       

In [19]:
# Comentarios por estudiante
comentarios

,Est_ID,Anio,CantComentarios
0,1.0,2021,13.0
1,2.0,2022,21.0
2,3.0,2019,0.0
3,5.0,2020,7.0
4,6.0,2021,4.0
...,...,...,...
2320,2713.0,2022,8.0
2321,2714.0,2022,1.0
2322,2715.0,2022,5.0
2323,2717.0,2022,14.0


In [20]:
# Columnas de actividades
actividades.columns

Index(['Est_ID', 'Actividad', 'Seccion', 'Ciclo', 'zona', 'Anio',
       'TipoCalificacion', 'PuntosPosibles', 'FechaTodoElDia',
       'FechaVencimiento', 'TipoEntrega', 'Nota', 'FechaCalificacion',
       'FechaEnvio', 'TipoEnvio', 'CantComentarios', 'NotaPtos',
       'NotaPorciento', 'imputado', 'residuo_estudiante', 'entrega_tarde',
       'es_examen', 'fecha_para_mes', 'mes', 'ptos_act', 'posibles_act'],
      dtype='object')

In [21]:
# Características mensuales en formato largo
mensual

,Est_ID,Anio,Seccion,mes,NotaPtos_mes,Nota_posible_mes,ptos_act_mes,posibles_act_mes,entregas_tarde_mes,porcentaje_acum_actividades,entregas_tarde_acum
0,1.0,2021,170,1,5.245,5.5,5.245,5.5,1,95.363636,1
1,1.0,2021,170,2,16.880,21.5,9.480,11.5,1,86.617647,2
2,1.0,2021,170,3,22.330,28.5,14.930,18.5,0,83.535211,2
3,1.0,2021,170,4,18.930,21.0,10.130,11.0,1,85.559140,3
4,1.0,2021,170,5,18.720,23.5,7.470,8.5,1,85.918182,4
...,...,...,...,...,...,...,...,...,...,...,...
11620,2718.0,2019,80,1,6.440,7.5,6.440,7.5,0,85.866667,0
11621,2718.0,2019,80,2,11.140,14.0,2.940,4.0,0,81.565217,0
11622,2718.0,2019,80,3,26.990,32.0,18.490,22.0,0,83.194030,0
11623,2718.0,2019,80,4,14.700,16.5,5.700,6.5,0,83.925000,0


In [22]:
# Columnas del registro
df_registro.columns

Index(['Est_ID', 'PAA', 'Colegio', 'TituloNivelMedio', 'NotaFinal', 'Aprobado',
       'Genero', 'EstadoCurso', 'veces_curso', 'Anio', 'Seccion_x',
       'porcentaje_acum_actividades_mes_1',
       'porcentaje_acum_actividades_mes_2',
       'porcentaje_acum_actividades_mes_3',
       'porcentaje_acum_actividades_mes_4',
       'porcentaje_acum_actividades_mes_5', 'entregas_tarde_acum_mes_1',
       'entregas_tarde_acum_mes_2', 'entregas_tarde_acum_mes_3',
       'entregas_tarde_acum_mes_4', 'entregas_tarde_acum_mes_5',
       'nota_examen_mes_2', 'nota_examen_mes_3', 'nota_examen_mes_4',
       'nota_examen_mes_5', 'delta_actividades_mes_2',
       'delta_actividades_mes_3', 'delta_actividades_mes_4',
       'delta_actividades_mes_5', 'z_porcentaje_acum_actividades_mes_1',
       'z_porcentaje_acum_actividades_mes_2',
       'z_porcentaje_acum_actividades_mes_3',
       'z_porcentaje_acum_actividades_mes_4',
       'z_porcentaje_acum_actividades_mes_5', 'z_nota_examen_mes_2',
       

In [24]:
# Cantidad de títulos de nivel medio distintos
df_registro['TituloNivelMedio'].nunique()

58

In [25]:
# Valores faltantes
df_registro.isna().sum()

Est_ID                                   0
PAA                                      0
Colegio                                  0
TituloNivelMedio                         0
NotaFinal                                0
Aprobado                                 0
Genero                                   0
EstadoCurso                              0
veces_curso                              0
Anio                                     0
Seccion_x                                0
porcentaje_acum_actividades_mes_1        0
porcentaje_acum_actividades_mes_2        0
porcentaje_acum_actividades_mes_3        0
porcentaje_acum_actividades_mes_4        0
porcentaje_acum_actividades_mes_5        0
entregas_tarde_acum_mes_1                0
entregas_tarde_acum_mes_2                0
entregas_tarde_acum_mes_3                0
entregas_tarde_acum_mes_4                0
entregas_tarde_acum_mes_5                0
nota_examen_mes_2                        0
nota_examen_mes_3                        0
nota_examen

### Valores faltantes y categorías poco frecuentes
Revisamos cuántos estudiantes hay por colegio: muchos colegios tienen menos de 10 estudiantes.

In [26]:
# Estudiantes por colegio
cuenta_colegio = df_registro['Colegio'].value_counts()

In [27]:
# Colegios con menos de 10 estudiantes
cuenta_colegio[cuenta_colegio < 10]

Colegio
COLEGIO SANTA MARIA DE MONTESCLAROS                                                         9
COLEGIO SAN FRANCISCO JAVIER DE LA VERAPAZ                                                  9
COLEGIO BOSTON                                                                              9
COLEGIO COBAN                                                                               9
COLEGIO PRIVADO MIXTO EVANGELICO ELIM                                                       9
                                                                                           ..
INSTITUTO PRIVADO MIXTO DE EDUCACION DIVERSIFICADA CENTRO DE ESTUDIOS DIVERSIFICADOS CED    1
CENTRO DE ESTUDIOS DE INFORMATICA COMPUTACION Y TECNOLOGIA CEICT                            1
COL MIXTO SAN JOSE                                                                          1
INST PARTICULAR SAN IGNACIO                                                                 1
TECNOLOGICO DON BOSCO                               

Los estudiantes sin `DiferenciaPromedio` son todos de 2019: ese año casi no hay fechas de entrega registradas, por lo que no se puede calcular.

In [28]:
# Estudiantes sin velocidad de calificación
sospechosos = df_registro[df_registro['DiferenciaPromedio'].isna()]

In [29]:
# Años de esos estudiantes
sospechosos['Anio'].unique()

array([2019])

In [30]:
# Actividades de esos estudiantes en 2019
actividades[(actividades['Est_ID'].isin(sospechosos['Est_ID'].unique())) & (actividades['Anio'] == 2019)]

,Est_ID,Actividad,Seccion,Ciclo,zona,Anio,TipoCalificacion,PuntosPosibles,FechaTodoElDia,FechaVencimiento,...,NotaPtos,NotaPorciento,imputado,residuo_estudiante,entrega_tarde,es_examen,fecha_para_mes,mes,ptos_act,posibles_act
0,635.0,EVALUACION ACUMULATIVA,10,1,True,2019,percent,14.0,NaT,NaT,...,10.08,72.0,NaN,NaN,0,0,2019-04-01,4,10.08,14.0
1,1419.0,EVALUACION ACUMULATIVA,10,1,True,2019,percent,14.0,NaT,NaT,...,9.80,70.0,NaN,NaN,0,0,2019-04-01,4,9.80,14.0
2,781.0,EVALUACION ACUMULATIVA,10,1,True,2019,percent,14.0,NaT,NaT,...,13.44,96.0,NaN,NaN,0,0,2019-04-01,4,13.44,14.0
3,853.0,EVALUACION ACUMULATIVA,10,1,True,2019,percent,14.0,NaT,NaT,...,10.50,75.0,NaN,NaN,0,0,2019-04-01,4,10.50,14.0
4,160.0,EVALUACION ACUMULATIVA,10,1,True,2019,percent,14.0,NaT,NaT,...,8.96,64.0,NaN,NaN,0,0,2019-04-22,4,8.96,14.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10453,2341.0,VERIFICACION 8,90,1,True,2019,percent,2.0,NaT,NaT,...,1.90,95.0,NaN,NaN,0,0,2019-05-12,5,1.90,2.0
10454,2603.0,VERIFICACION 8,90,1,True,2019,percent,2.0,NaT,NaT,...,1.90,95.0,NaN,NaN,0,0,2019-05-12,5,1.90,2.0
10455,738.0,VERIFICACION 8,90,1,True,2019,percent,2.0,NaT,NaT,...,2.00,100.0,NaN,NaN,0,0,2019-05-12,5,2.00,2.0
10456,2404.0,VERIFICACION 8,90,1,True,2019,percent,2.0,NaT,NaT,...,1.40,70.0,NaN,NaN,0,0,2019-05-12,5,1.40,2.0


In [31]:
# Columnas del registro
df_registro.columns

Index(['Est_ID', 'PAA', 'Colegio', 'TituloNivelMedio', 'NotaFinal', 'Aprobado',
       'Genero', 'EstadoCurso', 'veces_curso', 'Anio', 'Seccion_x',
       'porcentaje_acum_actividades_mes_1',
       'porcentaje_acum_actividades_mes_2',
       'porcentaje_acum_actividades_mes_3',
       'porcentaje_acum_actividades_mes_4',
       'porcentaje_acum_actividades_mes_5', 'entregas_tarde_acum_mes_1',
       'entregas_tarde_acum_mes_2', 'entregas_tarde_acum_mes_3',
       'entregas_tarde_acum_mes_4', 'entregas_tarde_acum_mes_5',
       'nota_examen_mes_2', 'nota_examen_mes_3', 'nota_examen_mes_4',
       'nota_examen_mes_5', 'delta_actividades_mes_2',
       'delta_actividades_mes_3', 'delta_actividades_mes_4',
       'delta_actividades_mes_5', 'z_porcentaje_acum_actividades_mes_1',
       'z_porcentaje_acum_actividades_mes_2',
       'z_porcentaje_acum_actividades_mes_3',
       'z_porcentaje_acum_actividades_mes_4',
       'z_porcentaje_acum_actividades_mes_5', 'z_nota_examen_mes_2',
       

In [32]:
# Valores faltantes
df_registro.isna().sum()

Est_ID                                   0
PAA                                      0
Colegio                                  0
TituloNivelMedio                         0
NotaFinal                                0
Aprobado                                 0
Genero                                   0
EstadoCurso                              0
veces_curso                              0
Anio                                     0
Seccion_x                                0
porcentaje_acum_actividades_mes_1        0
porcentaje_acum_actividades_mes_2        0
porcentaje_acum_actividades_mes_3        0
porcentaje_acum_actividades_mes_4        0
porcentaje_acum_actividades_mes_5        0
entregas_tarde_acum_mes_1                0
entregas_tarde_acum_mes_2                0
entregas_tarde_acum_mes_3                0
entregas_tarde_acum_mes_4                0
entregas_tarde_acum_mes_5                0
nota_examen_mes_2                        0
nota_examen_mes_3                        0
nota_examen

**Decisión:** imputar `DiferenciaPromedio` con la mediana, para no perder a los estudiantes de 2019.

In [33]:
# Imputación de la velocidad de calificación con la mediana
df_registro['DiferenciaPromedio'] = df_registro['DiferenciaPromedio'].fillna(
    df_registro['DiferenciaPromedio'].median()
)

In [34]:
# Verificación de valores faltantes
df_registro.isna().sum()

Est_ID                                 0
PAA                                    0
Colegio                                0
TituloNivelMedio                       0
NotaFinal                              0
Aprobado                               0
Genero                                 0
EstadoCurso                            0
veces_curso                            0
Anio                                   0
Seccion_x                              0
porcentaje_acum_actividades_mes_1      0
porcentaje_acum_actividades_mes_2      0
porcentaje_acum_actividades_mes_3      0
porcentaje_acum_actividades_mes_4      0
porcentaje_acum_actividades_mes_5      0
entregas_tarde_acum_mes_1              0
entregas_tarde_acum_mes_2              0
entregas_tarde_acum_mes_3              0
entregas_tarde_acum_mes_4              0
entregas_tarde_acum_mes_5              0
nota_examen_mes_2                      0
nota_examen_mes_3                      0
nota_examen_mes_4                      0
nota_examen_mes_

**Decisión:** agrupar en la categoría `OTRO` los colegios y títulos de nivel medio con menos de 10 estudiantes (611 estudiantes quedan con colegio `OTRO` y 125 con título `OTRO`). Con categorías tan pequeñas, el *target encoding* que se aplica en el modelado sería poco confiable y propenso al sobreajuste. Esta agrupación solo usa la frecuencia de cada categoría, no la variable objetivo, por lo que no introduce fuga de información.

In [35]:
def add_generic_category(array, text):
    """Devuelve OTRO si la categoría está en la lista de categorías poco frecuentes"""

    if text in array:
        return 'OTRO'
    return text

In [36]:
# Colegios y títulos con menos de 10 estudiantes
colegio_cuenta = df_registro['Colegio'].value_counts()
colegio_cuenta = colegio_cuenta[colegio_cuenta < 10]
titulo_cuenta = df_registro['TituloNivelMedio'].value_counts()
titulo_cuenta = titulo_cuenta[titulo_cuenta < 10]

In [37]:
# Agrupación de colegios poco frecuentes en OTRO
df_registro['Colegio'] = df_registro['Colegio'].apply(
    lambda x: add_generic_category(list(colegio_cuenta.index), x)
)

In [38]:
# Estudiantes por colegio después de agrupar
df_registro['Colegio'].value_counts()

Colegio
OTRO                                                                                     611
COLEGIO SUIZO AMERICANO                                                                  135
COLEGIO MIXTO CAPOUILLIEZ                                                                 94
LICEO JAVIER                                                                              84
CENTRO ESCOLAR EL ROBLE                                                                   67
COLEGIO VALLE VERDE                                                                       62
INSTITUTO PRIVADO DE EDUCACION DIVERSIFICADA LICEO GUATEMALA                              60
CENTRO EDUCATIVO TECNICO LABORAL KINAL                                                    57
COLEGIO INTERNACIONAL MONTESSORI                                                          54
COLEGIO DECROLY AMERICANO                                                                 49
COLEGIO GUATEMALTECO BILINGUE                                 

In [39]:
# Colegios agrupados
colegio_cuenta.index

Index(['COLEGIO SANTA MARIA DE MONTESCLAROS',
       'COLEGIO SAN FRANCISCO JAVIER DE LA VERAPAZ', 'COLEGIO BOSTON',
       'COLEGIO COBAN', 'COLEGIO PRIVADO MIXTO EVANGELICO ELIM',
       'COLEGIO JULIO VERNE', 'CENTRO ESCOLAR KYOOL',
       'CENTRO EDUCATIVO EL VALLE', 'COLEGIO MONTANO PORTAL LOS ALAMOS',
       'COLEGIO CAMPO ALTO',
       ...
       'COLEGIO EL PILAR', 'UNIVERSITY OF BELIZE',
       'COLEGIO ADVENTISTA EL PROGRESO',
       'INSTITUTO GUATEMALTECO DE EDUCACION RADIOFONICA IGER',
       'COMPLEJO EDUCATIVO BILINGUE DEL SUROCCIDENTE',
       'INSTITUTO PRIVADO MIXTO DE EDUCACION DIVERSIFICADA CENTRO DE ESTUDIOS DIVERSIFICADOS CED',
       'CENTRO DE ESTUDIOS DE INFORMATICA COMPUTACION Y TECNOLOGIA CEICT',
       'COL MIXTO SAN JOSE', 'INST PARTICULAR SAN IGNACIO',
       'TECNOLOGICO DON BOSCO'],
      dtype='object', name='Colegio', length=244)

In [40]:
# Agrupación de títulos poco frecuentes en OTRO
df_registro['TituloNivelMedio'] = df_registro['TituloNivelMedio'].apply(
    lambda x: add_generic_category(list(titulo_cuenta.index), x)
)

In [41]:
# Estudiantes por título después de agrupar
df_registro['TituloNivelMedio'].value_counts()

TituloNivelMedio
BACHILLER EN CIENCIAS Y LETRAS                                         1525
BACHILLER EN CIENCIAS Y LETRAS CON ORIENTACION EN COMPUTACION           362
OTRO                                                                    125
BACHILLER EN COMPUTACION                                                 65
BACHILLER EN COMPUTACION CON ORIENTACION CIENTIFICA                      39
BACHILLER BILINGUE EN CIENCIAS                                           29
BACHILLERATO EN CIENCIAS Y LETRAS EN COMPUTACION                         23
DIPLOMA NIVEL MEDIO 3RO BASICO                                           22
HIGH SCHOOL                                                              21
PERITO EN COMPUTACION                                                    18
BACHILLERATO EN CIENCIAS Y LETRAS CON ORIENTACION EN DISENO GRAFICO      17
BACHILLER INTERNACIONAL                                                  16
BACHILLER ACADEMICO                                                    

In [42]:
# Años del registro
df_registro['Anio']

0       2019
1       2019
2       2019
3       2019
4       2019
        ... 
2320    2022
2321    2020
2322    2021
2323    2021
2324    2022
Name: Anio, Length: 2325, dtype: int64

### Virtualidad
Los años 2020 y 2021 se impartieron en modalidad virtual por la pandemia de COVID-19. Creamos una variable binaria (`Virtualidad = 1`) para poder evaluar el efecto de la modalidad.

In [43]:
def anio_a_virtualidad(anio):
    """Devuelve 1 si el curso fue virtual (2020 y 2021) y 0 si fue presencial"""
    if anio == 2021.0 or anio == 2020.0:
        return 1 #1 si es virtual
    return 0

# Variable de modalidad del curso
df_registro['Virtualidad'] = df_registro['Anio'].apply(anio_a_virtualidad)


Limpiamos las columnas duplicadas por los *merge* y guardamos el dataset final.

In [44]:
# Limpieza de columnas duplicadas por los merge
df_registro = df_registro.rename(columns={'Seccion_x': 'Seccion'}).drop(columns='Seccion_y')

In [45]:
# Exportación del dataset de modelado
df_registro.to_csv('../../Datos/notas_pensamiento_cuantitativo.csv', index=False)

Distribución final de la variable objetivo: 2,010 aprobados y 315 reprobados (13.5 %).